# Metric Tests

## Setup

### Imports

In [1]:
import logging

# Local functions and classes
from types_and_classes import *
from utilities import *
from contour_plotting import *
from debug_tools import *
from structure_set import *
from relations import *
from region_slice import empty_structure


INFO:metrics.base:Registered calculator: orthogonal_margins (OrthogonalMarginsCalculator)
INFO:metrics.base:Registered calculator: minimum_margin (MinimumMarginCalculator)
INFO:metrics.base:Registered calculator: maximum_margin (MaximumMarginCalculator)
INFO:metrics.base:Registered calculator: minimum_distance (MinimumDistanceCalculator)


### Global Settings

In [2]:
%matplotlib inline

## Distance Tests


### Disjoint Boxes
![Disjoint Boxes](<../../Images/FreeCAD Images/Disjoint Boxes.png>)

In [3]:
def disjoint_boxes_example():
    slice_spacing = 0.1
    # Body structure defines slices in use
    body = make_vertical_cylinder(roi_num=0, radius=20, length=20, offset_z=0,
                                  spacing=slice_spacing)
    # embedded boxes
    left_cube = make_box(roi_num=1, width=2, offset_x=-3,
                         spacing=slice_spacing)
    right_cube = make_box(roi_num=2, width=2, offset_x=3,
                         spacing=slice_spacing)
    # combine the contours
    slice_data = left_cube + right_cube + body
    return slice_data


slice_data = disjoint_boxes_example()
disjoint_structures = StructureSet(slice_data)
structure_a = disjoint_structures.structures[1]
structure_b = disjoint_structures.structures[2]
relation = structure_a.relate(structure_b)
relation_type = relation.identify_relation()

print(f'\nleft_cube {relation_type.label} right_cube')
assert relation_type .relation_type == 'DISJOINT'

# Calculate minimum distance between the two boxes (ROI 1 and ROI 2)
distance_result = disjoint_structures.calculate_metric(1, 2, 'minimum_distance')

# Display results
print(f"Minimum Distance: {distance_result.minimum_distance:.2f} cm")

# Expected: Distance should be 4 cm (boxes centered at x=-3 and x=3, each with width 2)
# Distance = 3 - 1 - (1 - (-3)) = 6 - 2 = 4 cm
expected_distance = 4.0
assert abs(distance_result.minimum_distance - expected_distance) < 0.01, \
    f"Expected {expected_distance:.2f} cm, got {distance_result.minimum_distance:.2f} cm"
print(f"\n✓ Distance matches expected value of {expected_distance:.2f} cm")

INFO:structure_set:Adding structure Structure_0 (0)


INFO:structure_set:Adding structure Structure_1 (1)
INFO:structure_set:Adding structure Structure_2 (2)
INFO:structure_set:Calculating relationships for 3 structures
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_1 (ROI 1) as: Contains
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_2 (ROI 2) as: Contains
INFO:structure_set:Calculated relationships between Structure_1 (ROI 1) and Structure_2 (ROI 2) as: is Disjoint from
INFO:structure_set:Calculating logical flags for relationships
INFO:structure_set:Logical flag calculation complete
INFO:metrics.config:Loading metrics configuration from d:\OneDrive - Queen's University\Python\Projects\StructureRelations\src\metrics\metrics_config.json
INFO:metrics.config:Metrics configuration loaded successfully



left_cube is Disjoint from right_cube
Minimum Distance: 4.00 cm

✓ Distance matches expected value of 4.00 cm


#### Disjoint Concentric Cylinders
- Centred vertical cylinder with two cylinders on the same axis, positioned above and below.
- Regular contours are 2 slices apart (2 cm gap).
- Boundary slices (extrapolated ±0.5 slice spacing) are 1 cm apart.

![Disjoint Concentric Cylinders](<../../Images/FreeCAD Images/Disjoint Concentric Cylinders.png>)


In [4]:
def disjoint_concentric_cylinders_example():
    slice_spacing = 1
    # Body structure defines slices in use
    body = make_vertical_cylinder(roi_num=0, radius=12, length=10,
                                  spacing=slice_spacing)
    # Centred cylinder
    # Regular contours: z=-4 to z=4 (9 slices)
    # Boundary slices: z=-4.5 and z=4.5 (extrapolated ±0.5 slice spacing)
    primary_cylinder = make_vertical_cylinder(roi_num=1, radius=3, length=8,
                                              offset_z=0,
                                              spacing=slice_spacing)
    # Upper cylinder with SAME radius (3 cm) for pure vertical distance
    # cylinder 2 slices above primary: z=6 to z=10 (first regular contour at z=6)
    # Gap between regular contours: 6-4 = 2 slices
    # Gap between boundary slices: (6-0.5) - (4+0.5) = 1 cm
    upper_cylinder1 = make_vertical_cylinder(roi_num=2, radius=3, length=4,
                                             offset_z=8,
                                             spacing=slice_spacing)
    # Lower cylinder with SAME radius (3 cm)
    # cylinder 2 slices below primary: z=-10 to z=-6 (last regular at z=-6)
    # Gap between regular contours: (-4)-(-6) = 2 slices
    # Gap between boundary slices: (-4-0.5) - (-6+0.5) = 1 cm
    lower_cylinder2 = make_vertical_cylinder(roi_num=2, radius=3, length=4,
                                             offset_z=-8,
                                             spacing=slice_spacing)
    # combine the contours
    slice_data = body + primary_cylinder + upper_cylinder1 + lower_cylinder2
    return slice_data


slice_data = disjoint_concentric_cylinders_example()
structures = StructureSet(slice_data)
structure_a = structures.structures[1]
structure_b = structures.structures[2]
relation = structure_a.relate(structure_b)
relation_type = relation.identify_relation()

print(f'\nupper_and_lower_Cylinders {relation_type.label} primary_cylinder')
assert relation_type .relation_type == 'DISJOINT'

# Calculate minimum distance between the two boxes (ROI 1 and ROI 2)
distance_result = structures.calculate_metric(1, 2, 'minimum_distance')
# Display results
print(f"Minimum Distance: {distance_result.minimum_distance:.2f} cm")

# Verify: Solid-structure distance calculation
# Expected: Distance should be 1 cm (boundary slices are 1 cm apart)
# Primary cylinder regular contours: z=-4 to z=4
# Upper/Lower cylinder regular contours: z=6 to z=10 and z=-10 to z=-6
# Regular contour gap: 2 slices (2 cm)
# Boundary slice gap: 1 cm (boundaries extrapolate ±0.5 slice spacing)
# Primary boundary: z=4.5, Upper boundary: z=5.5, gap = 1.0 cm
# For SOLID structures, distance is measured between filled polygons, not just boundaries
expected_distance = 1.0
print(f"Expected: {expected_distance} cm")
assert abs(distance_result.minimum_distance - expected_distance) < 0.01, \
    f"Expected {expected_distance:.2f} cm, got {distance_result.minimum_distance:.2f} cm"
print(f"\n✓ Distance matches expected value of {expected_distance:.2f} cm")

INFO:structure_set:Adding structure Structure_0 (0)
INFO:structure_set:Adding structure Structure_1 (1)
INFO:structure_set:Adding structure Structure_2 (2)
INFO:structure_set:Calculating relationships for 3 structures
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_2 (ROI 2) as: Borders
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_1 (ROI 1) as: Contains
INFO:structure_set:Calculated relationships between Structure_2 (ROI 2) and Structure_1 (ROI 1) as: is Disjoint from
INFO:structure_set:Calculating logical flags for relationships
INFO:structure_set:Logical flag calculation complete



upper_and_lower_Cylinders is Disjoint from primary_cylinder
Minimum Distance: 1.00 cm
Expected: 1.0 cm

✓ Distance matches expected value of 1.00 cm


#### Vertically Disjoint Cylinders and Ring
- Centred hollow vertical cylinder with two cylinders on the same axis, positioned above and below.
- Regular contours are 2 slices apart (2 cm gap).
- Boundary slices (extrapolated ±0.5 slice spacing) are 1 cm apart.
- Ring and circle gap is 1 cm, so overall distance is $\sqrt{1^2 + 1^2} = 1.41$ cm.

![Vertically Disjoint Cylinders and Ring (Not Yet Created)](<../../Images/FreeCAD Images/Vertically Disjoint Cylinders and Ring.png>)


In [5]:
def vertically_disjoint_cylinders_and_ring_example():
    slice_spacing = 1
    # Body structure defines slices in use
    body = make_vertical_cylinder(roi_num=0, radius=12, length=10,
                                  spacing=slice_spacing)
    # Centred cylinder
    # Regular contours: z=-4 to z=4 (9 slices)
    # Boundary slices: z=-4.5 and z=4.5 (extrapolated ±0.5 slice spacing)
    primary_cylinder = make_vertical_cylinder(roi_num=1, radius=5, length=8,
                                              offset_z=0,
                                              spacing=slice_spacing)
    # Hole in the primary cylinder with smaller radius (4 cm) to create a ring
    # structure
    cylinder_hole = make_vertical_cylinder(roi_num=1, radius=4, length=8,
                                              offset_z=0,
                                              spacing=slice_spacing)
    # Upper cylinder with SAME radius (3 cm) for pure vertical distance
    # cylinder 2 slices above primary: z=6 to z=10 (first regular contour at z=6)
    # Gap between regular contours: 6-4 = 2 slices
    # Gap between boundary slices: (6-0.5) - (4+0.5) = 1 cm
    upper_cylinder1 = make_vertical_cylinder(roi_num=2, radius=3, length=4,
                                             offset_z=8,
                                             spacing=slice_spacing)
    # Lower cylinder with SAME radius (3 cm)
    # cylinder 2 slices below primary: z=-10 to z=-6 (last regular at z=-6)
    # Gap between regular contours: (-4)-(-6) = 2 slices
    # Gap between boundary slices: (-4-0.5) - (-6+0.5) = 1 cm
    lower_cylinder2 = make_vertical_cylinder(roi_num=2, radius=3, length=4,
                                             offset_z=-8,
                                             spacing=slice_spacing)
    # combine the contours
    slice_data = body + primary_cylinder + cylinder_hole + upper_cylinder1 + lower_cylinder2
    return slice_data


slice_data = vertically_disjoint_cylinders_and_ring_example()
structures = StructureSet(slice_data)
structure_a = structures.structures[1]
structure_b = structures.structures[2]
relation = structure_a.relate(structure_b)
relation_type = relation.identify_relation()

print(f'\nupper_and_lower_Cylinders {relation_type.label} primary_cylinder')
assert relation_type .relation_type == 'DISJOINT'

# Calculate minimum distance between the two boxes (ROI 1 and ROI 2)
distance_result = structures.calculate_metric(1, 2, 'minimum_distance')
# Display results
print(f"Minimum Distance: {distance_result.minimum_distance:.2f} cm")

# Verify: Solid-structure distance calculation
# Expected: Distance should be 1 cm (boundary slices are 1 cm apart)
# Primary cylinder regular contours: z=-4 to z=4
# Upper/Lower cylinder regular contours: z=6 to z=10 and z=-10 to z=-6
# Regular contour gap: 2 slices (2 cm)
# Boundary slice gap: 1 cm (boundaries extrapolate ±0.5 slice spacing)
# Primary boundary: z=4.5, Upper boundary: z=5.5, gap = 1.0 cm
# Because the primary cylinder is hollow, with a 1 cm horizontal gap
# the overall distance between the structures will be $\sqrt{1^2 + 1^2} = 1.41$ cm
expected_distance = sqrt(1**2 + 1**2)  # ≈ 1.41 cm
print(f"Expected: {expected_distance:.2f} cm")
# Tolerance of 0.02 cm (0.2 mm) accounts for polygon discretization
assert abs(distance_result.minimum_distance - expected_distance) < 0.02, \
    f"Expected {expected_distance:.2f} cm, got {distance_result.minimum_distance:.2f} cm"
print(f"\n✓ Distance matches expected value of {expected_distance:.2f} cm")

INFO:structure_set:Adding structure Structure_0 (0)
INFO:structure_set:Adding structure Structure_1 (1)
INFO:structure_set:Adding structure Structure_2 (2)
INFO:structure_set:Calculating relationships for 3 structures
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_1 (ROI 1) as: Contains
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_2 (ROI 2) as: Borders
INFO:structure_set:Calculated relationships between Structure_1 (ROI 1) and Structure_2 (ROI 2) as: is Disjoint from
INFO:structure_set:Calculating logical flags for relationships
INFO:structure_set:Logical flag calculation complete



upper_and_lower_Cylinders is Disjoint from primary_cylinder
Minimum Distance: 1.40 cm
Expected: 1.41 cm

✓ Distance matches expected value of 1.41 cm


#### Vertically Disjoint Different Sized Cylinders and Ring
- CSame as above except the the lower cylinder is smaller in diameter.
- This confirms that distance selected the minimum values from the two Z gaps..

![Vertically Disjoint Cylinders and Ring (Not Yet Created)](<../../Images/FreeCAD Images/Vertically Disjoint Cylinders and Ring.png>)


In [6]:
def vertically_disjoint_cylinders_and_ring_example():
    slice_spacing = 1
    # Body structure defines slices in use
    body = make_vertical_cylinder(roi_num=0, radius=12, length=10,
                                  spacing=slice_spacing)
    # Centred cylinder
    # Regular contours: z=-4 to z=4 (9 slices)
    # Boundary slices: z=-4.5 and z=4.5 (extrapolated ±0.5 slice spacing)
    primary_cylinder = make_vertical_cylinder(roi_num=1, radius=5, length=8,
                                              offset_z=0,
                                              spacing=slice_spacing)
    # Hole in the primary cylinder with smaller radius (4 cm) to create a ring
    # structure
    cylinder_hole = make_vertical_cylinder(roi_num=1, radius=4, length=8,
                                              offset_z=0,
                                              spacing=slice_spacing)
    # Upper cylinder with SAME radius (3 cm) for pure vertical distance
    # cylinder 2 slices above primary: z=6 to z=10 (first regular contour at z=6)
    # Gap between regular contours: 6-4 = 2 slices
    # Gap between boundary slices: (6-0.5) - (4+0.5) = 1 cm
    upper_cylinder1 = make_vertical_cylinder(roi_num=2, radius=3, length=4,
                                             offset_z=8,
                                             spacing=slice_spacing)
    # Lower cylinder with SAME radius (3 cm)
    # cylinder 2 slices below primary: z=-10 to z=-6 (last regular at z=-6)
    # Gap between regular contours: (-4)-(-6) = 2 slices
    # Gap between boundary slices: (-4-0.5) - (-6+0.5) = 1 cm
    lower_cylinder2 = make_vertical_cylinder(roi_num=2, radius=1, length=4,
                                             offset_z=-8,
                                             spacing=slice_spacing)
    # combine the contours
    slice_data = body + primary_cylinder + cylinder_hole + upper_cylinder1 + lower_cylinder2
    return slice_data


slice_data = vertically_disjoint_cylinders_and_ring_example()
structures = StructureSet(slice_data)
structure_a = structures.structures[1]
structure_b = structures.structures[2]
relation = structure_a.relate(structure_b)
relation_type = relation.identify_relation()

print(f'\nupper_and_lower_Cylinders {relation_type.label} primary_cylinder')
assert relation_type .relation_type == 'DISJOINT'

# Calculate minimum distance between the two boxes (ROI 1 and ROI 2)
distance_result = structures.calculate_metric(1, 2, 'minimum_distance')
# Display results
print(f"Minimum Distance: {distance_result.minimum_distance:.2f} cm")

# Verify: Solid-structure distance calculation
# Expected: Distance should be 1 cm (boundary slices are 1 cm apart)
# Primary cylinder regular contours: z=-4 to z=4
# Upper/Lower cylinder regular contours: z=6 to z=10 and z=-10 to z=-6
# Regular contour gap: 2 slices (2 cm)
# Boundary slice gap: 1 cm (boundaries extrapolate ±0.5 slice spacing)
# Primary boundary: z=4.5, Upper boundary: z=5.5, gap = 1.0 cm
# Because the primary cylinder is hollow, with a 1 cm horizontal gap
# the overall distance between the structures will be $\sqrt{1^2 + 1^2} = 1.41$ cm
expected_distance = sqrt(1**2 + 1**2)  # ≈ 1.41 cm
print(f"Expected: {expected_distance:.2f} cm")
# Tolerance of 0.02 cm (0.2 mm) accounts for polygon discretization
assert abs(distance_result.minimum_distance - expected_distance) < 0.02, \
    f"Expected {expected_distance:.2f} cm, got {distance_result.minimum_distance:.2f} cm"
print(f"\n✓ Distance matches expected value of {expected_distance:.2f} cm")

INFO:structure_set:Adding structure Structure_0 (0)
INFO:structure_set:Adding structure Structure_1 (1)
INFO:structure_set:Adding structure Structure_2 (2)
INFO:structure_set:Calculating relationships for 3 structures
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_1 (ROI 1) as: Contains
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_2 (ROI 2) as: Borders
INFO:structure_set:Calculated relationships between Structure_1 (ROI 1) and Structure_2 (ROI 2) as: is Disjoint from
INFO:structure_set:Calculating logical flags for relationships
INFO:structure_set:Logical flag calculation complete



upper_and_lower_Cylinders is Disjoint from primary_cylinder
Minimum Distance: 1.40 cm
Expected: 1.41 cm

✓ Distance matches expected value of 1.41 cm


In [7]:
# Examine per-slice distances for detailed analysis
if distance_result.slice_distances:
    print("Per-Slice Distances:")
    for region_pair, slice_dists in distance_result.slice_distances.items():
        print(f"\nRegion pair {region_pair}:")
        for (slice_a, slice_b), dist in sorted(slice_dists.items()):
            print(f"  Slice {slice_a} to {slice_b}: {dist:.2f} cm")

Per-Slice Distances:

Region pair (0, 0):
  Slice -4.5 to -6.0: 3.30 cm
  Slice -4.5 to -5.5: 3.11 cm
  Slice -4.0 to -5.5: 3.30 cm

Region pair (0, 1):
  Slice 4.0 to 5.5: 1.79 cm
  Slice 4.5 to 5.5: 1.40 cm
  Slice 4.5 to 6.0: 1.79 cm

Region pair (1, 0):
  Slice -4.5 to -6.0: 3.30 cm
  Slice -4.5 to -5.5: 3.11 cm
  Slice -4.0 to -5.5: 3.30 cm

Region pair (1, 1):
  Slice 4.0 to 5.5: 1.79 cm
  Slice 4.5 to 5.5: 1.40 cm
  Slice 4.5 to 6.0: 1.79 cm


#### Extended Inner Cylinder
- Concentric hollow cylinder with an interior cylinder extending beyond the outer cylinder's hole by one slice

![Extended Inner Cylinder](<../../Images/FreeCAD Images/Extended Inner cylinder.png>)

In [8]:
def extended_inner_cylinder_example():
    slice_spacing = 1
    # Body structure defines slices in use
    body = make_vertical_cylinder(roi_num=0, radius=12, length=16, offset_z=0,
                                  spacing=slice_spacing,
                                  num_points=100)
    outer_cylinder = make_vertical_cylinder(roi_num=1, radius=6, length=10,
                                            spacing=slice_spacing,
                                            num_points=100)
    cylinder_hole = make_vertical_cylinder(roi_num=1, radius=5, length=10,
                                           spacing=slice_spacing,
                                           num_points=100)
    inner_cylinder = make_vertical_cylinder(roi_num=2, radius=3, length=12,
                                            spacing=slice_spacing,
                                            num_points=100)

    # combine the contours
    slice_data = body + outer_cylinder + cylinder_hole + inner_cylinder
    return slice_data


slice_data = extended_inner_cylinder_example()
structures = StructureSet(slice_data)
structure_a = structures.structures[1]
structure_b = structures.structures[2]
relation = structure_a.relate(structure_b)
relation_type = relation.identify_relation()

print(f'\nOuter Hollow Cylinder {relation_type.label} Inner Cylinder')
assert relation_type .relation_type == 'DISJOINT'
# Calculate minimum distance between the two boxes (ROI 1 and ROI 2)
distance_result = structures.calculate_metric(1, 2, 'minimum_distance')
# Display results
print(f"Minimum Distance: {distance_result.minimum_distance:.2f} cm")

# Verify: Solid-structure distance calculation
# Expected: Distance should be 2 cm (Gap between inner solid cylinder and outer
# cylinder)  Distances from boundary slices will be greater.
expected_distance = 2.0  # ≈ 1.41 cm
print(f"Expected: {expected_distance:.2f} cm")
# Tolerance of 0.02 cm (0.2 mm) accounts for polygon discretization
assert abs(distance_result.minimum_distance - expected_distance) < 0.02, \
    f"Expected {expected_distance:.2f} cm, got {distance_result.minimum_distance:.2f} cm"
print(f"\n✓ Distance matches expected value of {expected_distance:.2f} cm")

INFO:structure_set:Adding structure Structure_0 (0)
INFO:structure_set:Adding structure Structure_1 (1)
INFO:structure_set:Adding structure Structure_2 (2)
INFO:structure_set:Calculating relationships for 3 structures
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_1 (ROI 1) as: Contains
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_2 (ROI 2) as: Contains
INFO:structure_set:Calculated relationships between Structure_1 (ROI 1) and Structure_2 (ROI 2) as: is Disjoint from
INFO:structure_set:Calculating logical flags for relationships
INFO:structure_set:Logical flag calculation complete



Outer Hollow Cylinder is Disjoint from Inner Cylinder
Minimum Distance: 2.00 cm
Expected: 2.00 cm

✓ Distance matches expected value of 2.00 cm


#### Horizontal Disjoint Cylinders
- Same as above but with horizontal cylinders
- Cylindrical shell with an interior cylinder ending outside the larger 
    diameter cylinder
- The relationship is **Disjoint** because the Second cylinder extends beyond 
    the horizontal bounds of the First Structure, but doe not intersect the 
    First Structure.

In [9]:
def disjoint_horizontal_cylinder_example():
    slice_spacing = 1
    # Body structure defines slices in use
    body = make_vertical_cylinder(roi_num=0, radius=12, length=16, offset_z=0,
                                  spacing=slice_spacing)
    outer_cylinder = make_horizontal_cylinder(roi_num=1, radius=6, length=10,
                                              spacing=slice_spacing)
    cylinder_hole = make_horizontal_cylinder(roi_num=1, radius=5, length=10,
                                             spacing=slice_spacing)
    surrounded_cylinder = make_horizontal_cylinder(roi_num=2, radius=3,
                                                   length=12,
                                                   spacing=slice_spacing)
    # combine the contours
    slice_data = body + outer_cylinder + cylinder_hole + surrounded_cylinder
    return slice_data


slice_data = disjoint_horizontal_cylinder_example()
structures = StructureSet(slice_data)
structure_a = structures.structures[1]
structure_b = structures.structures[2]
relation = structure_a.relate(structure_b)
relation_type = relation.identify_relation()

print(f'\nOuter Hollow Cylinder {relation_type.label} Inner Cylinder')
assert relation_type .relation_type == 'DISJOINT'
# Calculate minimum distance between the two boxes (ROI 1 and ROI 2)
distance_result = structures.calculate_metric(1, 2, 'minimum_distance')
# Display results
print(f"Minimum Distance: {distance_result.minimum_distance:.2f} cm")

# Verify: Solid-structure distance calculation
# Expected: Distance should be 2 cm (Gap between inner solid cylinder and outer
# cylinder)  Distances from boundary slices will be greater.
expected_distance = 2.0  # ≈ 1.41 cm
print(f"Expected: {expected_distance:.2f} cm")
# Tolerance of 0.02 cm (0.2 mm) accounts for polygon discretization
assert abs(distance_result.minimum_distance - expected_distance) < 0.02, \
    f"Expected {expected_distance:.2f} cm, got {distance_result.minimum_distance:.2f} cm"
print(f"\n✓ Distance matches expected value of {expected_distance:.2f} cm")

INFO:structure_set:Adding structure Structure_0 (0)
INFO:structure_set:Adding structure Structure_1 (1)
INFO:structure_set:Adding structure Structure_2 (2)
INFO:structure_set:Calculating relationships for 3 structures
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_1 (ROI 1) as: Contains
INFO:structure_set:Calculated relationships between Structure_0 (ROI 0) and Structure_2 (ROI 2) as: Contains
INFO:structure_set:Calculated relationships between Structure_1 (ROI 1) and Structure_2 (ROI 2) as: is Disjoint from
INFO:structure_set:Calculating logical flags for relationships
INFO:structure_set:Logical flag calculation complete



Outer Hollow Cylinder is Disjoint from Inner Cylinder
Minimum Distance: 1.00 cm
Expected: 2.00 cm


AssertionError: Expected 2.00 cm, got 1.00 cm

In [10]:
# Examine per-slice distances for detailed analysis
if distance_result.slice_distances:
    print("Per-Slice Distances:")
    for region_pair, slice_dists in distance_result.slice_distances.items():
        print(f"\nRegion pair {region_pair}:")
        for (slice_a, slice_b), dist in sorted(slice_dists.items()):
            print(f"  Slice {slice_a} to {slice_b}: {dist:.2f} cm")

Per-Slice Distances:

Region pair (0, 0):

Region pair (1, 0):
  Slice -4.0 to -3.5: 1.57 cm
  Slice -3.5 to -3.5: 1.74 cm
  Slice -3.5 to -3.0: 1.81 cm
  Slice -3.0 to -3.5: 2.05 cm
  Slice -3.0 to -3.0: 1.99 cm
  Slice -2.0 to -2.0: 1.17 cm
  Slice -1.0 to -1.0: 1.04 cm
  Slice 0.0 to 0.0: 1.00 cm
  Slice 1.0 to 1.0: 1.04 cm
  Slice 2.0 to 2.0: 1.17 cm
  Slice 3.0 to 3.0: 1.99 cm
  Slice 3.0 to 3.5: 2.05 cm
  Slice 3.5 to 3.0: 1.81 cm
  Slice 3.5 to 3.5: 1.74 cm
  Slice 4.0 to 3.5: 1.57 cm

Region pair (2, 0):
  Slice -4.0 to -3.5: 1.57 cm
  Slice -3.5 to -3.5: 1.74 cm
  Slice -3.5 to -3.0: 1.81 cm
  Slice -3.0 to -3.5: 2.05 cm
  Slice -3.0 to -3.0: 1.99 cm
  Slice -2.0 to -2.0: 1.17 cm
  Slice -1.0 to -1.0: 1.04 cm
  Slice 0.0 to 0.0: 1.00 cm
  Slice 1.0 to 1.0: 1.04 cm
  Slice 2.0 to 2.0: 1.17 cm
  Slice 3.0 to 3.0: 1.99 cm
  Slice 3.0 to 3.5: 2.05 cm
  Slice 3.5 to 3.0: 1.81 cm
  Slice 3.5 to 3.5: 1.74 cm
  Slice 4.0 to 3.5: 1.57 cm

Region pair (3, 0):
